[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/boltzgen/advanced/09_nanobody/09_nanobody_lab.ipynb)


# 09 — 나노바디(VHH) 실습

> 본문 [`09_nanobody.md`](09_nanobody.md) 와 **한 절씩 짝지어** 보세요.
> 이 노트북의 표·그래프·수치는 **여러분이 직접 돌린 결과**(`my_run/`)에서 계산합니다.
> **① 직접 설계 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서로 갑니다. 설계 셀은 NVIDIA GPU 전용이에요(CPU 폴백 없음) — Colab 이면 **런타임 → 런타임 유형 변경 → T4 GPU**.

## 0) 준비 — 저장소 클론 & 작업 폴더 이동

이 셀이 저장소를 클론하고 `09_nanobody/` 로 이동합니다. 로컬에서 `09_nanobody/` 안에 열었다면 클론 없이 진행돼요.

In [ ]:
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # fork 했다면 본인 주소로 바꾸세요
CLONE_AS = "bio-guides"
CHAPTER  = "09_nanobody"
PIP_PKGS = "pandas matplotlib"   # 없으면 설치할 분석 라이브러리

import os, sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules

# HF 가중치 다운로드가 멈춘 채 매달리지 않도록 타임아웃을 겁니다.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

_MARK = "boltzgen_viz.py"          # 이 파일이 있는 폴더가 advanced/ 루트

def _find_root():
    """advanced/ 루트를 찾습니다."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):   # 클론 직후: cwd 아래로만 깊이 3까지
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "advanced/ 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)          # 이 챕터 폴더로 이동 → data/ 상대경로가 그대로 동작
sys.path.insert(0, str(ADV_ROOT))     # boltzgen_viz import 보장
import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")

# import 안 되는 패키지만 설치합니다.
import importlib, importlib.util
_IMPORT_NAME = {"scikit-learn": "sklearn", "pillow": "PIL", "biopython": "Bio"}
def _have(mod):
    try: return importlib.util.find_spec(mod) is not None
    except Exception: return False
def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p))]
    if miss:
        print("필요 라이브러리 설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))  # python -m pip (Ch.03 권고)
        importlib.invalidate_caches()
if PIP_PKGS:
    _ensure(PIP_PKGS)

# 내 결과는 my_run/ 에 쌓이고, 없으면 커밋된 레퍼런스로 폴백합니다.
MY  = pathlib.Path("my_run")
MY.mkdir(exist_ok=True)

def find_one(pattern, ref, quiet=False):
    """my_run/ 에서 먼저 찾고, 없으면 레퍼런스 폴더에서 찾습니다."""
    for base in (MY/"final_ranked_designs", MY/"intermediate_designs_inverse_folded", MY):
        hits = sorted(base.glob(pattern))
        if hits:
            if not quiet: print(f"[내 결과]   {hits[0]}")
            return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"{ref} 에서 '{pattern}' 을 찾지 못했습니다."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def cols_in(df, *names):
    """내 실행 결과와 레퍼런스는 컬럼 구성이 조금 다를 수 있어, 있는 컬럼만 고른다."""
    missing = [c for c in names if c not in df.columns]
    if missing:
        print("(이 실행에는 없는 컬럼 — 건너뜁니다:", ", ".join(missing) + ")")
    return [c for c in names if c in df.columns]

def inherit_run(*rel_paths):
    """앞 챕터에서 돌린 설계 결과를 이어받습니다(없으면 레퍼런스로 폴백)."""
    global MY
    if (MY / "final_ranked_designs").exists():
        return MY
    for rel in rel_paths:
        p = pathlib.Path(rel)
        if (p / "final_ranked_designs").exists():
            print("[이어받기] 앞 챕터에서 직접 돌린 결과를 사용합니다:", p)
            MY = p
            return MY
    return MY

# 레퍼런스 그림을 덮어쓰지 않도록 my_ 접두어.
def my_fig(name):
    return f"my_{name}"

print("작업 폴더 :", pathlib.Path.cwd())

## 1) 직접 돌려보기 — 내 결과 만들기

- 학습용 규모 `num_designs=8 --budget=4` (레퍼런스 결과는 num_designs=30)
- 소요 시간 실측 **585초**(최종 4개) — **24GB GPU · 가중치 캐시** 기준이에요. Colab T4 는 가속 커널이 꺼져 더 걸리고(T4 실측치 없음), 첫 실행은 가중치 ~6GB 다운로드가 더 붙어요.
- 건너뛰면 아래 분석이 커밋된 레퍼런스 결과로 이어집니다.

In [ ]:
SPEC, PROTOCOL = "example/nanobody_against_penguinpox/penguinpox.yaml", "nanobody-anything"
NUM_DESIGNS, BUDGET = 8, 4

import shutil
OUT = MY.resolve()

def _gpu():
    try:
        import torch
        return torch.cuda.is_available()
    except ImportError:
        return shutil.which("nvidia-smi") is not None

if not _gpu():
    print("GPU 런타임이 아니라 설계 실행을 건너뜁니다 — 아래 분석은 레퍼런스 결과로 이어집니다.")
    print("  · 직접 돌리려면 Colab [런타임 → 런타임 유형 변경 → T4 GPU] 후 이 셀을 다시 실행하세요.")
else:
    SRC = ADV_ROOT / ".boltzgen_src"          # 예제 명세·타깃 CIF 가 들어 있는 BoltzGen 소스
    if not SRC.exists():
        _run(f'git clone --depth 1 https://github.com/HannesStark/boltzgen.git "{SRC}"')
    if not _have("boltzgen"):
        _run(f'"{sys.executable}" -m pip -q install -e "{SRC}"')
    try:
        _run(f'cd "{SRC}" && boltzgen run {SPEC} --output "{OUT}" --protocol {PROTOCOL} '
             f'--num_designs {NUM_DESIGNS} --budget {BUDGET}')
        print("\n내 결과 →", OUT / "final_ranked_designs")
    except Exception as e:
        print("\n설계 실행이 도중에 멈췄어요 —", type(e).__name__)
        print("  · 이 셀을 다시 실행하면 같은 --output 산출물을 재사용해 이어서 끝냅니다(실측 763초 → 재개 486초).")
        print("  · GPU 메모리가 부족했다면 NUM_DESIGNS 4, BUDGET 2 로 줄이세요.")

## 2) 무엇이 실제로 돌았나 — `steps.yaml` (본문 9.2)

`nanobody-anything` 은 `antibody-anything` 과 **내부 설정이 같아요** — inverse folding 에서 Cys 자동 금지,
design_folding 생략. Ch.08 에서 본 규칙이 사슬 하나짜리 VHH 에도 그대로 걸립니다.

레퍼런스를 만든 명령은 이거예요. 타깃은 penguinpox 단백질(`9bkq` B 체인)이고 복합체는 317~340 토큰이에요.

```bash
boltzgen run example/nanobody_against_penguinpox/penguinpox.yaml --output workbench/nanobody \
  --protocol nanobody-anything --num_designs 30 --budget 10
```

In [ ]:
steps_file = find_one("steps.yaml", "data/nanobody")
names = [ln.split(":", 1)[1].strip()
         for ln in steps_file.read_text().splitlines()
         if ln.strip().startswith("- name:")]

print("이 출력 폴더에서 실행된 스텝", len(names), "개")
for i, n in enumerate(names, 1):
    print(f"  {i}. {n}")
print()
print("design_folding 포함 —", "예" if "design_folding" in names else "아니오 (nanobody-anything 은 이 스텝을 생략)")
print("이 목록은 '이 출력 폴더에서 실제로 돈 스텝'이에요 — 앞 스텝 산출물을 재사용해 이어 돌리면 그만큼만 남습니다.")

## 3) 결과 표 — 무엇을 보나 (본문 9.4)

나노바디도 항체와 같은 메트릭군을 봐요 — `design_to_target_iptm`(타깃 결합, 순위를 좌우),
`design_ptm`·`filter_rmsd`(구조·자기일관성), `num_design`(재설계 CDR 영역 길이), `liability_*`(개발성).

In [ ]:
from boltzgen_viz import load_metrics, metrics_overview
import pandas as pd

CSV = str(find_one("final_designs_metrics_*.csv", "data/nanobody"))
df = pd.read_csv(CSV).sort_values("final_rank").reset_index(drop=True)
print("디자인", len(df), "개 | 전체 컬럼", df.shape[1], "종")
df[cols_in(df, "final_rank", "id", "design_ptm", "design_to_target_iptm",
           "filter_rmsd", "plip_hbonds_refolded", "num_design", "liability_score")]

## 4) 메트릭 그래프 — 순위와 함께 보기 (본문 9.6)

네 번째 패널은 순위별 인터페이스 H-bond 개수예요 — 결합면이 실제로 몇 개의 수소결합으로 잡혔는지
pTM·ipTM·RMSD 바와 나란히 봅니다.

In [ ]:
rows = load_metrics(CSV)
FIG = my_fig("09_nanobody_metrics.png")
metrics_overview(rows, "Nanobody (Penguinpox) — Design Metrics Overview", FIG, panel4="hbonds")
from IPython.display import Image; Image(FIG)

## 5) 어디를 재설계했나 — scaffold 5종과 `num_design` (본문 9.3·9.4)

레퍼런스에서는 rank 1 이 ipTM 0.252·RMSD 1.43Å 로 유일하게 두 축을 함께 만족했어요. 그 차이가 어디서 나왔는지
보려면 **무엇을 열어 뒀는지**부터 알아야 해요.

penguinpox 예제는 나노바디 scaffold **5종**을 함께 줍니다 — `7eow`(caplacizumab)·`7xl0`(vobarilizumab)·
`gontivimab`·`isecarosmab`·`sonelokimab`. 각 YAML 은 CDR1·CDR2·CDR3 를 **모두** `design` 으로 열고,
CDR3 는 `design_insertions` 로 길이까지 가변으로 둬요.

```yaml
# 7eow.yaml (caplacizumab, 요약)
design:            [ { chain: { id: B, res_index: 26..34,52..59,98..118 } } ]        # CDR1·CDR2·CDR3 모두 재설계
design_insertions: [ { insertion: { id: B, res_index: 98, num_residues: 1..14 } } ]  # CDR3 길이 가변
structure_groups:  # 골격은 visibility 2 로 유지, CDR 구간만 visibility 0 으로 자유
```

그래서 `num_design` 은 **CDR1+CDR2+CDR3 를 합친 재설계 구간 길이**예요.
흔한 오해가 여기서 나와요 — "CDR3 길이"는 BoltzGen 출력 컬럼이 **아니고**(Framework Identity·Humanness Score 도
마찬가지), CDR3 는 서열에서 직접 세야 해요(본문 9.4). 보존 Cys 다음부터 J-region(`WG?G…`) 직전까지가 CDR3 입니다.

In [ ]:
def j_region_start(s, window=25):
    """VHH J-region(WG?GT…) 시작 위치 — 못 찾으면 -1."""
    base = max(len(s) - window, 0)
    tail, hit = s[base:], -1
    for i in range(len(tail) - 3):
        if tail[i] == "W" and tail[i + 1] == "G" and tail[i + 3] == "G":
            hit = base + i
    return hit

def cdr3_of(s):
    """보존 Cys 3잔기 뒤 ~ J-region 직전 = CDR3 (Kabat 관례 근사)."""
    cys = [i for i, a in enumerate(s) if a == "C"]
    j = j_region_start(s)
    return s[cys[-1] + 3: j] if len(cys) >= 2 and j > 0 else ""

SEQ = (cols_in(df, "designed_chain_sequence", "sequence") or [None])[0]
has_nd = "num_design" in df.columns

lens = []
for _, r in df.iterrows():
    s = str(r[SEQ])
    cdr3 = cdr3_of(s)
    lens.append(len(cdr3))
    nd = f"num_design={int(r['num_design']):3d} | " if has_nd else ""
    print(f"rank{int(r['final_rank'])} {r['id']:14s} len={len(s):4d} | {nd}"
          f"CDR3={len(cdr3):2d}aa {cdr3}")

print()
if has_nd:
    print("num_design 범위", int(df["num_design"].min()), "~", int(df["num_design"].max()),
          "aa (CDR1+CDR2+CDR3 합계)")
print("서열에서 센 CDR3 길이 범위", min(lens), "~", max(lens), "aa")
print("→ 두 수가 다르면 정상이에요. num_design 을 CDR3 길이로 읽으면 안 됩니다 (본문 9.4).")

## 6) Developability — liability (본문 9.4)

CDR 이 얼마나 다양해졌는지 봤으니, 이제 그 서열이 **시약·약으로 버티는가**를 봐요. 나노바디도 산화·절단·응집 위험이
낮아야 대장균 발현부터 보관까지 문제가 없어요. 종합 지표 `liability_score` 는 **낮을수록** 개발하기 쉬운 후보고요.

모티프별 검출 내역은 `liability_*_count` 가 아니라 `liability_violations_summary` 문자열에 들어 있어요
(`ProtTrypx10(pos…); MetOx(pos…,sev5); …` 형태).

In [ ]:
import collections

score_cols = cols_in(df, "final_rank", "id", "liability_score", "liability_num_violations",
                     "liability_high_severity_violations", "liability_medium_severity_violations")
tbl = df[score_cols]
if "liability_score" in tbl.columns:
    tbl = tbl.sort_values("liability_score")
print("liability_score 낮은 순 (낮을수록 개발성 우수)")
print(tbl.to_string(index=False))

if "liability_violations_summary" in df.columns:
    tot = collections.Counter()
    for summary in df["liability_violations_summary"].astype(str):
        for item in summary.split(";"):
            head = item.strip().split("(", 1)[0]
            if not head:
                continue
            name, mult = head, 1
            if "x" in head:
                stem, _, tail = head.rpartition("x")
                if tail.isdigit():
                    name, mult = stem, int(tail)
            tot[name] += mult
    print()
    print("최종셋 전체에서 검출된 위험 모티프 (합계)")
    for name, n in tot.most_common():
        print(f"  {name:12s} {n}")

print()
print("→ ipTM 1위가 liability 1위는 아니에요. 두 순위가 어긋나는 지점이 8) 절의 선별 문제로 이어집니다.")

## 7) Framework 보존 — N말단·C말단 양쪽 (본문 9.5)

개발성까지 봤으니 마지막 확인은 "CDR 만 바뀐 게 맞나"예요. VHH framework 는 매우 보존적이라
앞은 `EVQLVESGGGLVQPG…` 로 시작하고 뒤는 `…WGQGTQVTVSS` 로 끝나요(본문 9.5).
앞만 보면 뒤쪽 J-region 이 무너진 디자인을 놓치니 두 끝을 모두 봅니다.

그리고 `nanobody-anything` 은 Cys 를 자동 금지하니, 사슬에 남는 Cys 는 framework 의 **보존 이황화쌍 2개**뿐이어야 해요.

In [ ]:
FW_N, FW_C = "EVQLVESGGGLVQPG", "WGQGTQVTVSS"

def diffs(seg, ref):
    """기준 모티프와 다른 자리를 '기준자리번호실제' 로."""
    return [f"{ref[i]}{i + 1}{seg[i]}" for i in range(min(len(seg), len(ref))) if seg[i] != ref[i]]

n_exact = c_exact = j_ok = cys_ok = 0
for _, r in df.iterrows():
    s = str(r[SEQ])
    head, tail = s[:len(FW_N)], s[-len(FW_C):]
    dn, dc = diffs(head, FW_N), diffs(tail, FW_C)
    n_exact += not dn
    c_exact += not dc
    j_ok += tail[:2] == "WG" and tail[3] == "G" and s.endswith("VTVSS")   # J-region 골격
    cys_ok += s.count("C") == 2
    print(f"rank{int(r['final_rank'])} {r['id']:14s} Cys={s.count('C')} "
          f"| N {head} {'일치' if not dn else '치환 ' + ','.join(dn)} "
          f"| C {tail} {'일치' if not dc else '치환 ' + ','.join(dc)}")

n = len(df)
print()
print(f"N말단 {FW_N} 완전 일치 {n_exact}/{n} · C말단 {FW_C} 완전 일치 {c_exact}/{n}")
print(f"J-region 골격(WG?GT…VTVSS) 유지 {j_ok}/{n} · 보존 Cys 2개 {cys_ok}/{n}")
print("→ J-region 골격이 살아 있고 Cys 가 2개면 CDR 만 바뀐 정상 그래프팅이에요 (자리 한두 개 치환은 같은 계열).")
print("  두 끝이 통째로 무너졌으면 scaffold YAML 의 not_design 범위를 넓혀 더 고정하세요 (본문 9.5).")

## 8) 어떤 후보를 실험으로 보낼까 (본문 9.6·9.7)

나노바디는 대장균 발현·His-tag 정제가 쉬워서 후보를 실험으로 빨리 넘길 수 있어요(본문 9.7).
그래서 선별이 곧 실험 비용이에요 — **ipTM 최고(결합) + liability_score 낮음(개발성) + RMSD 낮음(구조)**,
세 축 상위 3개의 교집합을 봅니다.

In [ ]:
axes = [("결합  ipTM 높음", "design_to_target_iptm", False),
        ("구조  RMSD 낮음", "filter_rmsd", True),
        ("개발성 liability_score 낮음", "liability_score", True)]

picks = []
for label, col, asc in axes:
    if col not in df.columns:
        continue
    top3 = df.sort_values(col, ascending=asc).head(3)
    picks.append(set(top3["id"]))
    shown = ", ".join(f"{i}({v:.3f})" if isinstance(v, float) else f"{i}({v})"
                      for i, v in zip(top3["id"], top3[col]))
    print(f"{label:26s} {shown}")

both = set.intersection(*picks) if picks else set()
print()
if both:
    print("세 축을 모두 만족하는 후보 —", ", ".join(sorted(both)))
    print("→ 이 후보부터 대장균 발현·SPR/BLI 로 검증하세요.")
else:
    print("세 축을 모두 만족하는 후보 없음 — 세 조건이 한 디자인으로 모이지 않았어요.")
    print("→ 표본을 키워 꼬리를 더 뽑아야 합니다 (본문 9.6).")

### 이 챕터에서 확인한 것 (본문 9.6)

- `nanobody-anything` 은 `antibody-anything` 과 같은 설정이고, `steps.yaml` 에 design_folding 이 없어요.
- scaffold 5종이 CDR1·CDR2·CDR3 를 모두 열어 두고 CDR3 는 길이까지 바꿔요. `num_design` 은 그 합계지
  CDR3 길이가 아니에요 — CDR3 는 서열에서 직접 셌어요.
- framework 는 N·C 양 끝이 기준 모티프와 거의 같고 Cys 는 2개뿐 — 정상 그래프팅이에요.
- ipTM 이 0.2 대에 머물고 RMSD 가 큰 디자인이 섞여 있는 건 실패가 아니라 **소표본(num_designs=30)** 탓이에요.
  penguinpox 는 어려운 타깃이라 실전이라면 `num_designs` 를 **2,000~10,000** 으로 올려야 ipTM 0.5+ 후보가
  꼬리에서 나와요(본문 9.6).

다음 Ch.10 은 상대가 단백질이 아니라 **소분자**일 때 파이프라인이 어떻게 달라지는지 봐요.